# Laboratorium: Wycena nieruchomości i prawdopodobieństwo wielowymiarowe. Kiedy klasyczne całki przestają działać?

W poprzednim etapie budowaliśmy modele, które przewidywały konkretne liczby lub decyzje (tak/nie). Dzisiaj wejdziemy głębiej – w świat **niepewności i rozkładów prawdopodobieństwa**. Jako Data Scientist w portalu zajmującym się inteligentną wyceną nieruchomości (jak Otodom czy Zillow), nie tylko chcesz podać cenę, ale też zrozumieć, jakie jest prawdopodobieństwo, że dana oferta pojawi się na rynku.

## Twoje cele na dziś:
1. **Analiza rozkładu łącznego 2D:** Zrozumiesz, jak dwie cechy (np. zarobki w okolicy i wielkość domu) współistnieją w przestrzeni matematycznej.
2. **Prawdopodobieństwo jako objętość:** Wykorzystasz **całkę podwójną**, aby obliczyć szansę na wystąpienie konkretnego profilu nieruchomości.
3. **Marginalizacja:** Dowiesz się, jak "usunąć" zbędne wymiary za pomocą całkowania po jednej zmiennej.
4. **Pokonanie "Przekleństwa Wymiarowości":** Zobaczysz, dlaczego przy 8 lub 30 cechach klasyczna analiza matematyczna zawodzi i dlaczego AI kocha **metodę Monte Carlo**.

---

## Od liceum do AI: Skąd tu się wzięły całki?

Z lekcji matematyki w liceum prawdopodobieństwo kojarzycie pewnie z rzutem kostką ($P(A) = \frac{n}{N}$). W analizie danych rzadko mamy jednak do czynienia ze skończoną liczbą wyników. Cena domu czy dochód to **zmienne ciągłe**.

* **W 1 wymiarze:** Prawdopodobieństwo, że zarobki wynoszą *dokładnie* 5423,12 zł jest równe zero. Dlatego pytamy o przedziały (np. od 5000 do 6000 zł). Matematycznie jest to **pole pod krzywą** (całka oznaczona).
* **W 2 wymiarach:** Jeśli chcemy znać szansę na to, że dom ma określoną liczbę pokoi **ORAZ** znajduje się w bogatej dzielnicy, szukamy części wspólnej. Na wykresie 3D nasze prawdopodobieństwo to już nie linia, a powierzchnia. Szukana szansa to **objętość pod tą powierzchnią**, którą obliczysz za pomocą **całki podwójnej**.



Dziś pracujemy na zbiorze **California Housing** (dostępnym w `scikit-learn`). Zawiera on dane o 20 640 blokach mieszkalnych w Kalifornii. Każdy z nich opisany jest przez 8 cech ciągłych. Przygotuj się – tam, gdzie kończy się wyobraźnia kartki papieru, zaczyna się potęga całkowania wielowymiarowego.

---

## Scenariusz Zadań

### Część 1: Rozkłady łączne – Prawdopodobieństwo w 2D (Całka Podwójna)
* **Zadanie biznesowe:** Bierzemy pod lupę dwie cechy: średni dochód w dzielnicy (`MedInc`) oraz średnią liczbę pokoi w domu (`AveRooms`). Estymujemy dla nich dwuwymiarowy rozkład prawdopodobieństwa. Szefostwo pyta: *"Jakie jest prawdopodobieństwo, że nowo powstająca dzielnica będzie miała średnio od 4 do 6 pokoi oraz dochód z przedziału X-Y?"*.
* **Podejście matematyczne i implementacja:** Zaimplementuj i oblicz całkę podwójną po zadanym prostokątnym obszarze z estymowanej funkcji gęstości 2D.

### Część 2: Marginalizacja (Zwijanie wymiarów)
* **Zadanie biznesowe:** Portal chce stworzyć ogólnokrajowy raport dotyczący wyłącznie zarobków, całkowicie ignorując to, jak duże są domy.
* **Podejście matematyczne i implementacja:** Masz do dyspozycji jedynie swój model łączny 2D (dochód i pokoje). Aby uzyskać rozkład samego dochodu, musisz zastosować proces **marginalizacji**, czyli scałkować swoją funkcję 2D po całej domenie liczby pokoi (od zera do nieskończoności). Matematycznie "zwijasz" jeden wymiar, uzyskując klasyczną krzywą Gaussa w 1D.

### Część 3: Twierdzenie Bayesa i Ściana Wymiarowości (Curse of Dimensionality)
* **Problem biznesowy:** Chcemy ocenić prawdopodobieństwo, że dana dzielnica należy do segmentu "Premium", biorąc pod uwagę **wszystkie 8 cech** ze zbioru danych (dochód, wiek, pokoje, sypialnie, populacja, obłożenie, współrzędne geograficzne).
* **Dyskusja analityczna:** Aby zastosować Twierdzenie Bayesa, mianownik wzoru (prawdopodobieństwo całkowite) wymaga policzenia **całki 8-krotnej**. Gdybyśmy użyli klasycznej metody siatek (np. metody trapezów z 1. semestru) i podzielili każdą z 8 osi na zaledwie 50 punktów, kod musiałby sprawdzić $50^8$, czyli około **39 miliardów punktów**! Policzenie jednej takiej całki zajęłoby klasycznym algorytmom stanowczo zbyt dużo czasu. W prawdziwym AI wymiarów potrafią być tysiące. To zjawisko nazywamy *Przekleństwem Wymiarowości*.

### Część 4: Całkowanie Monte Carlo na ratunek
* **Zadanie programistyczne:** Ponieważ metody numeryczne oparte na siatkach zawodzą, wdróż stochastyczne **Całkowanie Monte Carlo**.
* **Implementacja:** Napisz skrypt, który losuje $N$ punktów (np. 100 000 8-wymiarowych wektorów) z obszaru całkowania, uśrednia wartości funkcji gęstości i mnoży przez 8-wymiarową hiper-objętość. Porównaj czas wykonania z tradycyjnymi metodami. Wynik – dokładne przybliżenie całki wielokrotnej – powinien pojawić się na ekranie w ułamku sekundy.

### Zadanie 1: Czym naprawdę jest całka? Prawdopodobieństwo jako suma małych "klocków"

**Kontekst biznesowy:** Twój zespół analizuje szanse na to, że nowa dzielnica spełni określone kryteria:
1. **MedInc** (oś X, zarobki): od 3.0 do 5.0 (czyli 30 tys. - 50 tys. dolarów).
2. **AveRooms** (oś Y, pokoje): od 4.0 do 6.0 średnio na dom.

**Matematyka w ujęciu inżynieryjnym:**
Nasz Senior Data Scientist przygotował model (funkcję `funkcja_gestosci(x, y)`), który nad naszą płaską mapą (zarobki i pokoje) rozpościera "górę" prawdopodobieństwa (powierzchnię 3D). Całkowite prawdopodobieństwo wystąpienia takiej dzielnicy to po prostu **objętość pod tą górą** w zadanym obszarze.



Zamiast używać gotowych i niezrozumiałych funkcji z bibliotek, obliczymy tę objętość (czyli **całkę podwójną**) ręcznie, metodą numeryczną opartą na sumach Riemanna. Wyobraź sobie, że budujemy tę objętość z małych, ciasno ułożonych klocków (graniastosłupów).

---

### Instrukcja implementacji krok po kroku:

Twój kod (w komórce poniżej) wymaga uzupełnienia kilku kluczowych zmiennych. Oto co i jak musisz zaprogramować:

**KROK 1: Wymiary podstawy pojedynczego klocka (`krok_x`, `krok_y`)**
Podzieliliśmy cały nasz badany zakres osi X (od `x_min` do `x_max`) oraz osi Y (od `y_min` do `y_max`) na określoną liczbę równych części (zmienna `liczba_krokow`).
* **Co zrobić:** Oblicz długość jednego takiego kroku dla obu osi.
* **Jak zrobić:** Odejmij wartość początkową od końcowej i podziel przez `liczba_krokow`. Np. dla osi X będzie to `(x_max - x_min) / liczba_krokow`.

**KROK 2: Pole podstawy klocka (`pole_podstawy`)**
Każdy nasz mały klocek ma w podstawie prostokąt.
* **Co zrobić:** Oblicz pole tego prostokąta.
* **Jak zrobić:** Pomnóż szerokość (`krok_x`) przez długość (`krok_y`).

**KROK 3 i 4: Podwójna pętla i wysokość klocka (`wysokosc`, `objetosc_slupka`)**
Przygotowaliśmy dla Ciebie dwie pętle `for`, które "skanują" naszą mapę, przesuwając się kratka po kratce wzdłuż osi X i Y. Wewnątrz tej pętli stoisz na konkretnych współrzędnych `x` oraz `y`.
* **Co zrobić:** 1. Musisz sprawdzić, jak wysoki klocek prawdopodobieństwa należy tu postawić.
  2. Musisz policzyć objętość tego pojedynczego klocka.
* **Jak zrobić:** 1. Wywołaj funkcję `funkcja_gestosci(x, y)` – zwróci Ci ona dokładną wysokość. Przypisz ją do zmiennej `wysokosc`.
  2. Pomnóż `wysokosc` przez policzone wcześniej `pole_podstawy` i zapisz wynik w zmiennej `objetosc_slupka`.

**KROK 5: Całkowanie, czyli podsumowanie (`calkowita_objetosc`)**
Znak całki $\int \int$ to matematyczny symbol nieskończonego sumowania. My w Pythonie zrobimy to samo dodając do siebie objętości wszystkich klocków.
* **Co zrobić:** Zaktualizuj główny "koszyk" na objętość.
* **Jak zrobić:** Do zmiennej `calkowita_objetosc` dodaj obliczoną w danym obrocie pętli `objetosc_slupka` (możesz użyć operatora `+=`).

Jeśli zaprogramujesz to poprawnie, tysiące obrotów pętli zsumują maleńkie objętości w jeden, bardzo dokładny wynik całkowitego prawdopodobieństwa!

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal
from sklearn.datasets import fetch_california_housing

# ---------------------------------------------------------
# CZĘŚĆ UKRYTA: Przygotowanie modelu przez Senior Data Scientista
# (To nasza wyrocznia, która mówi, jak wysoki jest słupek w punkcie x, y)

# Pobieramy dane (dodane dla pewności, że komórka zadziała samodzielnie)
X = fetch_california_housing().data

X_2d = X[:, [0, 2]]
mean_2d = np.mean(X_2d, axis=0)
cov_2d = np.cov(X_2d, rowvar=False)
model_2d = multivariate_normal(mean_2d, cov_2d)

def funkcja_gestosci(x, y):
    """Zwraca wysokość 'góry prawdopodobieństwa' dla zarobków (x) i pokoi (y)."""
    return model_2d.pdf([x, y])
# ---------------------------------------------------------

# TODO
calkowita_objetosc=#TODO

print(f"Prawdopodobieństwo (całkowita objętość): {calkowita_objetosc:.4f} (czyli {calkowita_objetosc*100:.2f}%)")

Rozpoczynam ręczne całkowanie (budowanie 10000 klocków)...
Prawdopodobieństwo (całkowita objętość): 0.1292 (czyli 12.92%)


### Co właściwie widzimy na tym wykresie? Zrozumieć magię całkowania

Poniższy wykres 3D pokazuje wprost, co tak naprawdę wykonał Twój komputer w poprzednim zadaniu. Przeanalizujmy to krok po kroku, aby odczarować matematykę:

**1. Półprzezroczysta, kolorowa powierzchnia (Twoja "góra")**
To jest nasz idealny, matematyczny model prawdopodobieństwa dostarczony przez Senior Data Scientista (nasza funkcja gęstości). Im wyżej wznosi się ta powierzchnia w danym punkcie na płaszczyźnie, tym częściej w rzeczywistości występują w Kalifornii dzielnice o takich zarobkach i liczbie pokoi. Szukane przez biznes prawdopodobieństwo to **całkowita objętość** znajdująca się pod tą górą w naszym wyciętym fragmencie mapy.

**2. Niebieskie klocki (Twoje pętle `for` w akcji!)**
Komputer nie potrafi policzyć objętości pod gładką, nieregularną powierzchnią w magiczny sposób za pomocą jednego wzoru. Zamiast tego zrobił dokładnie to, co napisałeś w kodzie:
* Podzielił podłogę na siatkę małych prostokątów (to Twoje `pole_podstawy`).
* Dla każdego kwadracika zmierzył wysokość "góry" w tym punkcie (to Twoja zmienna `wysokosc`).
* Wybudował tam niebieski graniastosłup (to Twoja `objetosc_slupka`).
* Na końcu zsumował je wszystkie razem.

**Inżynieryjny kompromis (Zbieżność całki):**
Na naszym wykresie użyliśmy zaledwie 400 klocków (siatka 20x20). Jeśli przyjrzysz się uważnie, zauważysz, że przez to powstają "schodki", a klocki miejscami lekko wystają ponad powierzchnię lub jej nie dotykają. Oznacza to, że nasz wynik jest **numerycznym przybliżeniem**. Znak całki w matematyce oznacza, że dzielimy ten obszar na *nieskończenie wiele* nieskończenie małych kawałków.

Gdybyś w swoim kodzie zmienił zmienną opisującą liczbę kroków na 1000, pętla wybudowałaby milion mikroskopijnych klocków. Kształt schodków idealnie zrównałby się z powierzchnią, a wynik byłby perfekcyjnie dokładny. Tak właśnie w świecie AI radzimy sobie ze skomplikowaną matematyką – rozbijamy ją na miliony prostych operacji! W poniższej wizualizacji zmień liczbę kloców np. na 5x5, 10x10, 30x30 aby to lepiej zrozumieć.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import multivariate_normal

# ---------------------------------------------------------
# CZĘŚĆ 1: Przygotowanie modelu i danych (to co w poprzednich zadaniach)
# ---------------------------------------------------------
# Zakładamy, że zbiór danych X jest już załadowany (fetch_california_housing)
X_2d = X[:, [0, 2]]
mean_2d = np.mean(X_2d, axis=0)
cov_2d = np.cov(X_2d, rowvar=False)
model_2d = multivariate_normal(mean_2d, cov_2d)

# Granice naszego obszaru biznesowego
x_min, x_max = 3.0, 5.0  # Zarobki
y_min, y_max = 4.0, 6.0  # Pokoje

# ---------------------------------------------------------
# CZĘŚĆ 2: Ustawienia wizualizacji
# DLA CZYTELNOŚCI WYKRESU: Używamy rzadszej siatki (np. 20 krokow)
# ---------------------------------------------------------
liczba_krokow_wiz = 20

# Tworzymy punkty na osiach
x_coords = np.linspace(x_min, x_max, liczba_krokow_wiz)
y_coords = np.linspace(y_min, y_max, liczba_krokow_wiz)

# Tworzymy siatkę 2D (współrzędne każdego narożnika klocka)
X_grid, Y_grid = np.meshgrid(x_coords, y_coords)

# Obliczamy wymiary podstawy słupka
krok_x_wiz = (x_max - x_min) / liczba_krokow_wiz
krok_y_wiz = (y_max - y_min) / liczba_krokow_wiz

# ---------------------------------------------------------
# CZĘŚĆ 3: Tworzenie wykresu 3D
# ---------------------------------------------------------
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(projection='3d')

# --- 1. Rysujemy gładką powierzchnię gęstości (Model) ---
# Używamy gęstszej siatki tylko do narysowania powierzchni, żeby była gładka
x_smooth = np.linspace(x_min - 0.5, x_max + 0.5, 100)
y_smooth = np.linspace(y_min - 0.5, y_max + 0.5, 100)
X_smooth, Y_smooth = np.meshgrid(x_smooth, y_smooth)
Z_smooth = model_2d.pdf(np.dstack((X_smooth, Y_smooth)))

# Rysujemy powierzchnię z niskim alpha (półprzezroczystą), żeby widzieć słupki
ax.plot_surface(X_smooth, Y_smooth, Z_smooth, cmap='viridis', alpha=0.3, edgecolor='none')

# --- 2. Rysujemy "ręcznie" słupki (Numeryczne przybliżenie całki) ---
# Flattening (spłaszczanie) siatki, aby przejść przez każdy punkt pętlą (matplotlib bar3d tak wymaga)
x_flat = X_grid.ravel()
y_flat = Y_grid.ravel()
z_bottom = np.zeros_like(x_flat) # Słupki startują od poziomu z=0

# Obliczamy wysokość każdego słupka w punkcie (x, y)
# Używamy np.dstack, aby przygotować dane dla modelu_2d.pdf w formacie wektorowym
pos = np.dstack((x_flat, y_flat))
z_heights = model_2d.pdf(pos).ravel()

# Używamy bar3d do narysowania graniastosłupów
# dx, dy, dz to wymiary: szerokość, długość, wysokość
ax.bar3d(x_flat, y_flat, z_bottom, krok_x_wiz, krok_y_wiz, z_heights,
         color='skyblue', alpha=0.8, edgecolor='black', linewidth=0.5)

# --- 3. Ustawienia estetyczne ---
ax.set_title(f'Wizualizacja całki podwójnej (Suma Riemanna)\n'
             f'Przybliżenie za pomocą {liczba_krokow_wiz}x{liczba_krokow_wiz} słupków', fontsize=16)
ax.set_xlabel('Zarobki (MedInc)', fontsize=12)
ax.set_ylabel('Liczba Pokoi (AveRooms)', fontsize=12)
ax.set_zlabel('Gęstość prawdopodobieństwa', fontsize=12)

# Ustawiamy kąt widzenia, żeby dobrze widzieć przestrzeń pod powierzchnią
ax.view_init(elev=25, azim=135)

plt.tight_layout()
plt.show()

### Zadanie 2: Marginalizacja, czyli "zwijanie" wymiaru za pomocą plasterków

**Kontekst biznesowy:** Zarząd portalu nieruchomości chce raportu, który pokaże rozkład samych zarobków (`MedInc`) w Kalifornii. Problem? Twój zaawansowany model `funkcja_gestosci(x, y)` oczekuje podania dwóch wartości: zarobków (x) i liczby pokoi (y). Zarządu kompletnie nie interesuje liczba pokoi!

**Matematyka (Marginalizacja):**
Aby poznać łączną szansę na konkretne zarobki (np. $x = 4.0$), musisz zsumować szanse dla *wszystkich możliwych liczb pokoi* występujących przy tych zarobkach.
Matematycznie oznacza to policzenie całki pojedynczej (1D) z naszej funkcji po zmiennej $dy$:
$$f_{zarobki}(x) = \int f(x,y) \,dy$$

**Zrozumieć przez programowanie (Całkowanie ręczne):**
Nie użyjemy gotowych funkcji! Zrobimy to inżynieryjnie. Wyobraź sobie, że zatrzymujesz zmienną `x` (zarobki) w miejscu, a następnie idziesz wzdłuż osi `y` (pokoje). W każdym kroku mierzysz wysokość "góry" prawdopodobieństwa i budujesz płaski, wąski prostokąt. Suma pól tych prostokątów da Ci pożądany wynik (zmarginalizowaną gęstość).

**Twoje zadanie:**
Napisz funkcję `marginalna_gestosc_x_reczna(x)`, która dla zadanego, *nieruchomego* `x` policzy sumę pól prostokątów wzdłuż osi `y`. Przyjmiemy realistyczny zakres liczby pokoi od 0 do 20.

**Instrukcja krok po kroku:**
1. Zdefiniowaliśmy zakres dla pokoi: od `0.0` do `20.0` oraz liczbę kroków (`liczba_krokow_y = 200`). Oblicz szerokość pojedynczego paska (`krok_y`).
2. Stwórz pętlę `for`, która przejdzie przez wszystkie punkty na osi Y (użyj `np.linspace`).
3. Wewnątrz pętli zmierz wysokość w punkcie `(x, y)` za pomocą funkcji `funkcja_gestosci(x, y)`. Zauważ, że `x` cały czas pozostaje bez zmian!
4. Oblicz pole wąskiego paska (wysokość $\times$ szerokość) i dodaj je do całkowitej sumy.
5. Zwróć ostateczną sumę za pomocą `return`. Przygotowany przez nas kod pod spodem wygeneruje wykres, aby sprawdzić Twoje rozwiązanie.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("Rozpoczynam ręczną marginalizację (krojenie góry wzdłuż osi Y)...")

# Ustalamy zakres dla osi Y (pokoje), po którym będziemy sumować (całkować)
# W teorii powinniśmy całkować od -nieskończoności do +nieskończoności,
# ale domy z ujemną liczbą pokoi nie istnieją, a odcięcie na 20 wystarczy.
y_min_marg = 0.0
y_max_marg = 20.0
liczba_krokow_y = 200

def marginalna_gestosc_x_reczna(x):
    """Oblicza wartość rozkładu brzegowego dla zadanej wartości x (zarobków)."""

    #TODO
# ---------------------------------------------------------
# CZĘŚĆ WYKRESOWA
# ---------------------------------------------------------
print("Generowanie wykresu rozkładu brzegowego...")

# Generujemy 50 punktów dla osi zarobków (od 0 do 10)
os_zarobkow = np.linspace(0, 10, 50)

# Dla każdego punktu na osi X, wywołujemy Twoją funkcję, która w środku
# robi pętlę po osi Y (to w praktyce podwójna pętla!)
gestosc_zarobkow = [marginalna_gestosc_x_reczna(x) for x in os_zarobkow]

# Rysujemy wynikowy, "zgnieciony" wymiar 1D
plt.figure(figsize=(10, 5))
plt.plot(os_zarobkow, gestosc_zarobkow, color='green', linewidth=3)
plt.fill_between(os_zarobkow, gestosc_zarobkow, color='lightgreen', alpha=0.5)
plt.title("Rozkład Brzegowy (Marginalny) Zarobków\n(Uzyskany przez ręczne odcałkowanie liczby pokoi)", fontsize=14)
plt.xlabel("Mediana zarobków (MedInc w dziesiątkach tys. $)", fontsize=12)
plt.ylabel("Gęstość prawdopodobieństwa", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

### Część 3: Twierdzenie Bayesa i Ściana Wymiarowości (Curse of Dimensionality)

**Kontekst biznesowy:** Zarząd chce wiedzieć, jakie jest prawdopodobieństwo, że nowo badana dzielnica to segment "Premium". Tym razem jednak nie możemy opierać się tylko na zarobkach i pokojach. Musimy uwzględnić **wszystkie 8 cech** ze zbioru danych jednocześnie (dochód, wiek domów, pokoje, sypialnie, populacja, obłożenie, współrzędne geograficzne).

**Problem analityczny:** Aby odpowiedzieć na to pytanie przy użyciu **Twierdzenia Bayesa**, musimy policzyć tzw. prawdopodobieństwo całkowite (mianownik we wzorze Bayesa). Wymaga to obliczenia **całki 8-krotnej**!
Gdybyśmy spróbowali użyć metody siatek z poprzednich zadań (krojenia na plasterki lub budowania z klocków) i podzielili każdą z 8 osi na zaledwie 50 punktów, nasz kod musiałby sprawdzić i zsumować $50^8$, czyli około **39 miliardów punktów**! Twój komputer liczyłby to tygodniami. W prawdziwym sztucznym inteligencji wymiarów potrafią być tysiące. Ten fenomen paraliżu obliczeniowego nazywamy **Przekleństwem Wymiarowości** (ang. *Curse of Dimensionality*).

---

## **Część 4: Symulacja Monte Carlo – Jak ugryźć 8 wymiarów?**

W poprzednim zadaniu odkryliśmy, że obliczenie szansy na znalezienie "typowej" dzielnicy metodą siatki (sprawdzanie punkt po punkcie) zajęłoby wieczność. Dlaczego? Bo w 8 wymiarach przestrzeń jest tak ogromna, że większość obliczeń marnujemy na "puste" miejsca, gdzie prawdopodobieństwo wynosi 0.

### **Rozwiązanie: Metoda Monte Carlo (czyli "Rzut rzutką")**

Zamiast budować skomplikowaną siatkę, zachowamy się jak gracze w kasynie. Zamiast mierzyć całą podłogę milimetr po milimetrze, rozrzucimy na niej tysiące ziarenek piasku i sprawdzimy, ile z nich wpadło w interesujący nas obszar.

W statystyce AI nazywamy to **próbkowaniem (samplingiem)**. Zamiast liczyć trudną całkę analitycznie, generujemy $N=100\,000$ "wirtualnych dzielnic" i sprawdzamy ich parametry.

#### **Logika algorytmu: Co musisz zrobić?**

Wyobraź sobie, że zamykamy nasze dane w 8-wymiarowym "pudełku" (hiperprostopadłościanie). Szerokość tego pudełka na każdej z 8 osi to zakres naszych danych (od minimum do maksimum dla każdej cechy).

1.  **Przygotuj "pudełko":** Znajdź wartości min i max dla każdej z 8 cech (np. dochód od 0.5 do 15, wiek domu od 1 do 52 itd.).
2.  **Rzuć 100 000 razy:** Wygeneruj losowe punkty, które wpadną do tego pudełka. Każdy punkt to "wirtualna dzielnica" o 8 losowych cechach.
3.  **Sprawdź wysokość:** Dla każdego punktu zapytaj nasz model: *"Jak bardzo prawdopodobne jest, że taka dzielnica istnieje?"*.
4.  **Wyciągnij średnią:** Pomnóż objętość swojego 8-wymiarowego pudełka przez średnią z tych prawdopodobieństw.

Wzór, który realizujesz, to:
$$\text{Całka} \approx \text{Objętość Pudełka} \times \frac{1}{N} \sum_{i=1}^{N} f(x_i)$$

To wszystko! Właśnie zamieniłeś problem, który wymagałby miliardów obliczeń, na prostą symulację, która trwa ułamek sekundy.

---

### **Instrukcja do zadania programistycznego:**

1.  **Zdefiniuj zakresy:** Stwórz dwie zmienne `mins` i `maxs`, zawierające wartości minimalne i maksymalne dla wszystkich 8 kolumn z Twoich danych (wykorzystaj `X.min(axis=0)` oraz `X.max(axis=0)`).
2.  **Oblicz objętość (Volume):** To iloczyn różnic $(max - min)$ dla wszystkich 8 wymiarów.  
    *Podpowiedź: użyj `np.prod(maxs - mins)`.*
3.  **Wylosuj punkty:** Użyj funkcji `np.random.uniform(low=mins, high=maxs, size=(100000, 8))`, aby stworzyć macierz losowych "rzutek".
4.  **Oblicz wynik:** * Przekaż wylosowane punkty do funkcji `model_8d.pdf()`.
    * Oblicz ostateczny wynik: `volume * np.mean(prawdopodobienstwa)`.


### Rozgrzewka: Jak Monte Carlo mierzy objętość? (Przykład z 3D)

Zanim rzucimy się na 8-wymiarowe dane z Kalifornii, zrozummy na czym polega spryt metody **Monte Carlo**.

Wyobraź sobie, że zapomnieliśmy wzoru na objętość kuli, ale bardzo dobrze znamy wzór na objętość sześcianu (bok $\times$ bok $\times$ bok).
Zamykamy naszą kulę o promieniu $r=1$ w idealnie przylegającym do niej sześcianie o boku $a=2$. Objętość sześcianu wynosi więc $2 \times 2 \times 2 = 8$.

**Algorytm "Ślepego Rzucania Rzutkami":**
1. Zamykamy oczy i rzucamy tysiące rzutek losowo w stronę naszego sześcianu.
2. Po każdym rzucie sprawdzamy: *Czy rzutka wbiła się w kulę, czy poleciała w róg sześcianu (poza kulę)?* (Matematycznie sprawdzamy to z twierdzenia Pitagorasa: czy $x^2 + y^2 + z^2 \le r^2$).
3. Ponieważ rzucamy całkowicie losowo, odsetek rzutek, które trafiły w kulę, będzie odpowiadał odsetkowi objętości sześcianu, jaką ta kula zajmuje!

Wzór staje się dziecinnie prosty:
$$\text{Objętość Kuli} \approx \text{Objętość Sześcianu} \times \frac{\text{Liczba trafień w kulę}}{\text{Wszystkie rzuty}}$$

Uruchom poniższy kod. Skrypt najpierw policzy objętość używając 100 000 rzutek, a potem wyrysuje małą próbkę na wykresie 3D, abyś mógł to zobaczyć na własne oczy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

print("Rozpoczynam symulację Monte Carlo dla kuli 3D...")

# ---------------------------------------------------------
# 1. OBLICZENIA (Używamy dużej liczby punktów dla dokładności)
# ---------------------------------------------------------
N_obliczenia = 100
promien = 1.0

# Losujemy współrzędne x, y, z w przedziale od -1 do 1 (nasz sześcian)
# Generujemy od razu macierz o rozmiarze (N_obliczenia, 3)
punkty_xyz = np.random.uniform(low=-promien, high=promien, size=(N_obliczenia, 3))

# Obliczamy odległość każdego punktu od środka układu współrzędnych (x^2 + y^2 + z^2)
# np.sum(..., axis=1) sumuje wartości wzdłuż wierszy
odleglosci_do_kwadratu = np.sum(punkty_xyz**2, axis=1)

# Trafienie w kulę ma miejsce, gdy odległość^2 jest mniejsza lub równa promien^2
trafienia_w_kule = odleglosci_do_kwadratu <= promien**2
liczba_trafien = np.sum(trafienia_w_kule)

# Wzór Monte Carlo
objetosc_szescianu = (2 * promien) ** 3  # 2 * 2 * 2 = 8
estymowana_objetosc = objetosc_szescianu * (liczba_trafien / N_obliczenia)

# Dokładny wzór matematyczny dla porównania (4/3 * pi * r^3)
dokladna_objetosc = (4/3) * np.pi * (promien**3)

print(f"Liczba rzutów: {N_obliczenia}")
print(f"Trafienia w kulę: {liczba_trafien}")
print(f"Estymowana objętość kuli (Monte Carlo): {estymowana_objetosc:.5f}")
print(f"Dokładna objętość kuli (Wzór analityczny): {dokladna_objetosc:.5f}")
print(f"Błąd przybliżenia: {abs(estymowana_objetosc - dokladna_objetosc):.5f}")

# ---------------------------------------------------------
# 2. WIZUALIZACJA (Używamy małej liczby punktów, aby nie zawiesić komputera)
# ---------------------------------------------------------
N_wykres = 2_000

# Bierzemy tylko pierwsze 2000 punktów do narysowania
punkty_wykres = punkty_xyz[:N_wykres]
trafienia_wykres = trafienia_w_kule[:N_wykres]

# Rozdzielamy punkty na te w środku kuli i te na zewnątrz
punkty_wewnatrz = punkty_wykres[trafienia_wykres]
punkty_zewnatrz = punkty_wykres[~trafienia_wykres] # Tylda (~) oznacza logiczne zaprzeczenie

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(projection='3d')

# Rysujemy punkty wewnątrz kuli (niebieskie)
ax.scatter(punkty_wewnatrz[:, 0], punkty_wewnatrz[:, 1], punkty_wewnatrz[:, 2],
           color='dodgerblue', alpha=0.6, s=15, label='Wewnątrz kuli')

# Rysujemy punkty na zewnątrz kuli, ale w sześcianie (szare/czerwone)
ax.scatter(punkty_zewnatrz[:, 0], punkty_zewnatrz[:, 1], punkty_zewnatrz[:, 2],
           color='crimson', alpha=0.2, s=15, label='Poza kulą (w sześcianie)')

ax.set_title(f'Wizualizacja Monte Carlo\n(Wyświetlono {N_wykres} z {N_obliczenia} punktów)', fontsize=14)
ax.set_xlabel('Oś X')
ax.set_ylabel('Oś Y')
ax.set_zlabel('Oś Z')
ax.legend(loc='upper right')

# Ustawiamy równe proporcje osi, aby kula nie wyglądała jak jajko
ax.set_box_aspect([1,1,1])

plt.show()

Poniżej kod (przygotowany przez Senior Data Scientist), który dla wylosowanych punktów wyznaczy prawdopodobieństwo wystąpienia.

In [ ]:
from sklearn.neighbors import KernelDensity
from sklearn.datasets import fetch_california_housing
import numpy as np

# 1. Pobieramy dane (wszystkie 8 cech)
data = fetch_california_housing()
X = data.data  # Macierz 20640 wierszy x 8 kolumn

# 2. Tworzymy i trenujemy model gęstości (KDE)
# Używamy jądra gaussowskiego. Parametr bandwidth (szerokość pasma)
# decyduje o tym, jak bardzo "wygładzony" jest model.
model_8d = KernelDensity(kernel='gaussian', bandwidth=0.5).fit(X)

# ---  Jak działa ten obiekt? ---
# model_8d.sample(n) -> generuje n losowych punktów (dzielnic) z rozkładu
# model_8d.score_samples(X) -> zwraca logarytm gęstości (log-likelihood)
# model_8d.pdf -> (stworzymy ją poniżej dla wygody studentów)


def get_pdf(x):
    # score_samples zwraca log(pdf), więc używamy exp(), by wrócić do skali 0-1
    log_pdf = model_8d.score_samples(x)
    return np.exp(log_pdf)

# Nadpisujemy metodę,
model_8d.pdf = get_pdf

model_8d.rvs = model_8d.sample

print("Model 8D został zainicjalizowany i nauczony na danych z Kalifornii.")

In [ ]:

mins = X.min(axis=0) # minima dla każdej z 8 cech
maxs = X.max(axis=0) # maksima dla każdej z 8 cech

#Tutaj rozwiaz zadanie

##Klątwa Wymiarowości

Jeżeli kod daje wynik bliski 0, jest poprawny matematycznie, ale padł ofiarą zjawiska, które w Data Science nazywamy "Przekleństwem Wymiarowości" (Curse of Dimensionality). Choć objętość waszego "pudełka" jest gigantyczna, szansa na trafienie w nim w realne dane przy losowaniu jednostajnym jest bliska zeru.Oto dlaczego tak się stało:
1. **Problem "Pustego Kosmosu"** Wyobraźcie sobie, że szukacie igły w stogu siana.W 2 wymiarach (kartka papieru) igła zajmuje sporą część powierzchni. Łatwo w nią trafić losowo rzuconą "rzutką".W 3 wymiarach (pokój pełen siana) jest już trudniej, ale $100\,000$ rzutek pewnie w coś trafi.W 8 wymiarach (wasz przypadek) przestrzeń rośnie wykładniczo. Wasze "pudełko" (zakresy od minimum do maksimum) jest jak cały Układ Słoneczny, a realne dane (sensowne dzielnice Kalifornii) to zaledwie kilka małych kulek rozsypanych gdzieś w tej próżni.

2. **Statystyczne "Pudło"** Funkcja `np.random.uniform` losuje punkty całkowicie bezmyślnie – rozrzuca je równomiernie po całym waszym "kosmosie".Prawdziwe dane o domach nie są rozproszone równomiernie. Ludzie nie mieszkają w dzielnicach, gdzie zarobki wynoszą $0$, dom ma $500$ lat, a średnia liczba pokoi to $100$.Wasza funkcja pdf() mówi jasno: "Wysokość (prawdopodobieństwo) jest większa od zera tylko tam, gdzie te kombinacje cech mają sens".Skoro rzucaliście rzutkami losowo w 8-wymiarową próżnię, żadna z $100\,000$ rzutek nie trafiła w obszar, który model uznał za realny. Średnia wysokość wyniosła $0$, więc wynik całej całki to również $0$.

3. `Objętość to Pułapka` Zauważcie wartość zmiennej volume w waszym wydruku. To gigantyczna liczba! W 8 wymiarach "pudełko poszukiwań" jest tak ogromne, że nawet jeśli model ma tam jakieś "góry" prawdopodobieństwa, są one niewyobrażalnie wąskie w porównaniu do całego pudełka. Metoda Monte Carlo oparta na losowaniu jednostajnym po prostu ich nie zauważa.

### Jak to naprawić? (Spoiler do kolejnych zadań)

Aby pokonać tę klątwę, nie możemy strzelać w ciemno. Musimy użyć "inteligentnego próbkowania":Zamiast rzucać rzutkami w całe pudełko ('uniform') i oceniać je przez `pdf()`, poprosimy model, żeby sam wygenerował punkty tam, gdzie wie, że "coś jest" (używając metody `model_8d.sample()`).To tak, jakbyśmy zamiast szukać ludzi na chybił-trafił w całym kosmosie, po prostu wyciągnęli z modelu gotową listę adresów, pod którymi faktycznie ktoś mieszka.

## Wniosek:
 Metoda `pdf() + uniform` świetnie działa do liczenia całek w 2D lub 3D. W 8D i wyżej – matematyczna pustka zawsze wygrywa i musimy zmienić podejście na symulację!

Skoro przeszliśmy z metody „ślepych rzutek” (`np.random.uniform`) na „inteligentne próbkowanie” (`model_8d.sample`), nasz algorytm staje się znacznie bardziej skuteczny, a intuicja basenu zmienia się w intuicję sondowania terenu.

# 💡 Przewodnik: Jak obliczyć prawdopodobieństwo metodą „Sondowania Modelu”?

W świecie 8D obliczenie prawdopodobieństwa poprzez „strzelanie w ciemno” (losowanie jednostajne) kończy się wynikiem zero, bo przestrzeń jest zbyt pusta. Dlatego używamy inteligentnego próbkowania.

Zamiast szukać igły w stogu siana, prosimy model, aby sam podał nam igły do ręki.

---

# Nowy Algorytm: „Sondaż Populacji”

## Krok 1: Zdefiniuj „Sito” (Warunki)

Zamiast budować pudełko i liczyć jego objętość, określamy warunki logiczne, które nas interesują.

### Przykład

> „Szukamy miejsc, gdzie `MedInc > 6.0` oraz `HouseAge < 15`.”

---

## Krok 2: Generowanie „Wirtualnej Kalifornii”

Prosimy model, aby wygenerował ogromną liczbę punktów ($N$) tam, gdzie on sam „wie”, że dane mają sens.

Dzięki temu omijamy pusty kosmos i skupiamy się tylko na realnych scenariuszach.

```python
N = 100000

# Model sam wybiera punkty o wysokiej gęstości
# (pokonujemy klątwę wymiarowości!)
punkty = model_8d.sample(N)
```

## Krok 3: Filtrowanie (Nakładanie Sita)

Sprawdzamy, który z wygenerowanych punktów spełnia nasze warunki z Kroku 1.

Tworzymy tzw. maskę logiczną.

```python
# Przykład:
# sprawdzamy warunek dla kolumny nr 0 (dochód)
# i nr 1 (wiek)

maska = (punkty[:, 0] > 6.0) & (punkty[:, 1] < 15)
```

## Krok 4: Wynik Końcowy (Statystyczna Magia)

Prawdopodobieństwo to po prostu odsetek punktów, które przeszły przez nasze sito.

Nie potrzebujemy już liczyć `volume` ani używać `pdf` do wyznaczania wag, ponieważ model zrobił to za nas „pod maską” podczas losowania próbek.

```python
prawdopodobienstwo = np.mean(maska)
```

# 🚀 Co za chwilę zrobicie?

Za moment przetestujecie to podejście w 3 zadaniach.

Pamiętajcie:

1. Wygenerujcie raz dużą próbkę (`samples`).
2. Wybierzcie odpowiednie kolumny (indeksy `0–7`).
3. Policzcie średnią z warunku.

To właśnie Wasze prawdopodobieństwo!

### **🚀 Zadania Praktyczne: Zostań Analitykiem AI**

Wykorzystaj model `model_8d` oraz poznany algorytm "Skanera Przestrzeni", aby odpowiedzieć na poniższe pytania biznesowe. Wyniki przedstaw w procentach z dokładnością do czwartego miejsca po przecinku.

---

#### **Zadanie 4.1: Segment "Duże Domy"**
Zarząd chce oszacować, jak rzadkie są dzielnice, w których średnia liczba pokoi (`AveRooms`) jest **większa niż 5**, przy założeniu, że pozostałe parametry mogą być dowolne (w granicach danych).

* **Wskazówka:** Dla cechy `AveRooms` przyjmij zakres $[5, \text{max}]$, a dla pozostałych 7 cech ich pełny zakres $[\text{min}, \text{max}]$.

---

#### **Zadanie 4.2: Nowoczesne Budownictwo w Zasięgu Ręki**
Jaka jest szansa na znalezienie dzielnicy "młodej" (`HouseAge` < 15 lat), w której mediana dochodów (`MedInc`) jest powyżej przeciętnej (np. powyżej 6.0), ale zagęszczenie ludzi (`AveOccup`) nie przekracza 3 osób na dom?

---

#### **Zadanie 4.3: Wykrywanie Anomalii (Złoty Graal)**
Oblicz szansę na znalezienie dzielnicy, która łączy trzy bardzo rzadkie cechy:
1.  Bardzo wysokie zarobki (`MedInc` > 10.0).
2.  Dużą średnią liczbę pokoi (`AveRooms` > 6).
3.  Lokalizację w prestiżowym rejonie San Francisco (Szerokość `Latitude`: 37-38, Długość `Longitude`: od -123 do -122).

> **💡 Uwaga:** Jeśli Twój wynik wynosi $0.0000\%$, spróbuj zwiększyć liczbę próbek $N$ do $1\,000\,000$, aby sprawdzić, czy zjawisko jest całkowicie niemożliwe, czy po prostu ekstremalnie rzadkie w Twoim modelu.


In [ ]:
#TODO

# 🧠 Zadania Zaawansowane: Inteligentne Filtrowanie Danych

Teraz, gdy potraficie już używać „sita” na pojedynczych cechach, czas na wyższy poziom. Poniższe zadania wymagają stworzenia masek, które łączą cechy ze sobą za pomocą operacji matematycznych i złożonej logiki.

## 💡 Przypominajka — Indeksy Kolumn

| Indeks | Nazwa cechy | Opis |
|---|---|---|
| 0 | MedInc | Dochód |
| 1 | HouseAge | Wiek |
| 2 | AveRooms | Pokoje |
| 3 | AveBedrms | Sypialnie |
| 4 | Population | Populacja |
| 5 | AveOccup | Zagęszczenie |
| 6 | Latitude | Szerokość geograficzna |
| 7 | Longitude | Długość geograficzna |

---

## Zadanie 4.4: „Hotele i Akademiki” (Analiza Proporcji)

W typowym domu sypialnie stanowią tylko część wszystkich pomieszczeń. Zarząd portalu podejrzewa, że w danych ukrywają się budynki o nietypowej strukturze (np. hotele lub akademiki).

Oblicz prawdopodobieństwo znalezienia dzielnicy, w której sypialnie (`AveBedrms`) stanowią ponad 50% wszystkich pokoi (`AveRooms`).

## Zadanie 4.5: „Strefa Wpływów” (Filtrowanie Przestrzenne)

Klienci premium chcą mieszkać blisko Doliny Krzemowej.

Oblicz szansę na znalezienie dzielnicy leżącej w promieniu maksymalnie `0.5` stopnia geograficznego od centrum Silicon Valley (`Lat: 37.3`, `Lon: -122.0`), która dodatkowo charakteryzuje się bardzo młodą zabudową (`HouseAge < 10` lat).

In [ ]:
#TODO